# K06 — Held-out corpora, second batch: M4 (six languages), HC3-Chinese, DAIGT essays (stage S2/S5)

Builds three more feature tables under the frozen protocol and its addendum A1
(`docs/EXPERIMENT_PROTOCOL.md`, section 8):

* **M4**, non-English subsets: Arabic, Bulgarian, German, Indonesian, Russian, Urdu — about 42,000 documents;
* **HC3-Chinese**: three largest sources — about 12,000 documents;
* **DAIGT v2** (PERSUADE student essays + essays by several LLMs) — about 12,000 documents.

Non-English text: detector = XLM-R fine-tuned here on English M4 documents and applied
cross-lingually, plus an in-language character TF-IDF detector; outcomes = multilingual
sentiment and lexical diversity. DAIGT (English) is processed exactly like RAID and Beemo in K05.

**Before running**
1. *Add Input* → search **"DAIGT V2 Train Dataset"** (by *thedrcat*) → Add. (M4 and HC3-Chinese are downloaded by the notebook.)
2. Settings: *Accelerator* = GPU T4 x2, *Internet* = On.
3. Run the first four code cells by hand. The fourth must end with `SMOKE TEST OK`.
   If a format check prints a problem, send that output back before the long run.
4. *Save Version* → *Save & Run All (Commit)*.

**Expected GPU time:** about 1 hour for the two non-English corpora, 2–3 hours for DAIGT.
**Outputs:** `m4_features.parquet`, `hc3zh_features.parquet`, `daigt_features.parquet` and the
three `*_summary.json`. A corpus whose source is missing is skipped and reported at the end.
**Resume:** save the Output as a dataset, add it as Input, run again (files `k06_*`).

In [ ]:
!pip -q install -U transformers accelerate vaderSentiment textstat nrclex sentencepiece
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import nltk
nltk.download("cmudict", quiet=True); nltk.download("subjectivity", quiet=True); nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
SUBJ_DIR = nltk.data.find("corpora/subjectivity")
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPUs:", torch.cuda.device_count())
assert d2vi.__version__ >= "2.0.0.dev6"
assert torch.cuda.is_available(), "turn the GPU on in Settings"

In [ ]:
import glob, os, shutil, json, hashlib, time
import numpy as np, pandas as pd
WORK, RAW = "/kaggle/working", "/tmp/k06"
os.makedirs(RAW, exist_ok=True)
KEEP = ("_features.parquet", "_summary.json")
for f in glob.glob("/kaggle/input/**/*", recursive=True):            # resume from an earlier run
    name = os.path.basename(f)
    if os.path.isfile(f) and (name.startswith("k06_") or name.endswith(KEEP)) and "/k06_xlmr/" not in f:
        if not os.path.exists(f"{WORK}/{name}"):
            shutil.copy(f, f"{WORK}/{name}")
for d in glob.glob("/kaggle/input/**/k06_xlmr", recursive=True):     # the fine-tuned detector
    if os.path.isdir(d) and not os.path.exists(f"{WORK}/k06_xlmr"):
        shutil.copytree(d, f"{WORK}/k06_xlmr")
print("working dir:", sorted(os.listdir(WORK)))
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
STATUS, HASHES = {}, {}

In [ ]:
# Sources and format checks (CPU, a few minutes). Nothing is scored here.
from d2vi.data.multilingual import (M4_URL, M4_LANGUAGES, M4_ENGLISH, load_m4, load_m4_english,
                                    load_hc3_chinese, load_daigt)
try:
    names = [n for files in M4_LANGUAGES.values() for n in files.values()] + list(M4_ENGLISH.values())
    for n in names:
        if not os.path.exists(f"{RAW}/{n}"):
            !wget -q -O {RAW}/{n} {M4_URL}/{n}
        HASHES[n] = sha(f"{RAW}/{n}")
    m4, m4en = load_m4(RAW), load_m4_english(RAW)
    print(m4.groupby(["domain", "source"]).size().unstack(fill_value=0))
    print("English training documents:", m4en.groupby(["domain", "source"]).size().to_dict())
    assert set(m4.domain) == set(M4_LANGUAGES) and len(m4en) == 8000
    STATUS["m4"] = "ready"
except Exception as err:
    STATUS["m4"] = f"NOT AVAILABLE: {err!r}"[:400]
try:
    HC3ZH = f"{RAW}/hc3_chinese_all.jsonl"
    if not os.path.exists(HC3ZH):
        !wget -q -O {HC3ZH} https://huggingface.co/datasets/Hello-SimpleAI/HC3-Chinese/resolve/main/all.jsonl
    HASHES["hc3_chinese_all.jsonl"] = sha(HC3ZH)
    print("first record keys:", sorted(json.loads(open(HC3ZH, encoding="utf-8").readline())))
    hc3zh = load_hc3_chinese(HC3ZH)
    print(hc3zh.groupby(["domain", "source"]).size().unstack(fill_value=0))
    assert hc3zh.domain.nunique() == 3 and hc3zh.groupby(["domain", "source"]).size().min() >= 1000
    STATUS["hc3zh"] = "ready"
except Exception as err:
    STATUS["hc3zh"] = f"NOT AVAILABLE: {err!r}"[:400]
try:
    found = sorted(glob.glob("/kaggle/input/**/train_v2_drcat_02.csv", recursive=True))
    assert found, "Add Input -> 'DAIGT V2 Train Dataset' (thedrcat); file train_v2_drcat_02.csv not found"
    head = pd.read_csv(found[0], nrows=5)
    print("DAIGT columns:", list(head.columns))
    full = pd.read_csv(found[0], usecols=lambda c: c in ("label", "source", "prompt_name"))
    print(full.groupby(["label", "source"]).size().to_string())
    HASHES["train_v2_drcat_02.csv"] = sha(found[0])
    daigt = load_daigt(found[0])
    print(daigt.groupby("source").size().to_dict())
    assert daigt[daigt.a.eq(0)].shape[0] >= 2000 and daigt[daigt.a.eq(1)].source.nunique() >= 2
    STATUS["daigt"] = "ready"
except Exception as err:
    STATUS["daigt"] = f"NOT AVAILABLE: {err!r}"[:400]
print()
for k, v in STATUS.items():
    print(f"{k:6s} {v}")

In [ ]:
# Smoke test of the GPU code on a handful of documents (about 2 minutes, mostly downloads)
from d2vi.features.multilingual import (XLMRDetector, XLMRSentiment, score_multilingual,
                                        finalize_multilingual, XLMR_ID)
from d2vi.features.heldout import operating_points
from d2vi.features.pipeline import package_versions
assert STATUS["m4"] == "ready", "M4 is needed to train the cross-lingual detector"
tiny = m4en.groupby("a").head(48)
smoke = XLMRDetector(epochs=1, batch_size=16).fit(tiny.text.tolist(), tiny.a.to_numpy())
p = smoke.predict(tiny.text.tolist()[:8])
assert p.shape == (8,) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
SENT = XLMRSentiment()
s = SENT.predict(["I love this, it is wonderful!", "This is terrible and I hate it.",
                  "Это замечательно, мне очень нравится!", "这太糟糕了，我很讨厌。"])
print("sentiment (expected + - + -):", s.round(2))
assert s[0] > 0 > s[1], "sentiment labels are not in the expected order"
del smoke; torch.cuda.empty_cache()
print("SMOKE TEST OK")

In [ ]:
# Cross-lingual detector: XLM-R fine-tuned on English M4 documents (about 6 minutes on a T4)
MODEL_DIR = f"{WORK}/k06_xlmr"
check = m4en.groupby(["domain", "a"]).tail(125)                # 500 documents kept out, as a check only
train = m4en.drop(check.index)
if os.path.exists(f"{MODEL_DIR}/config.json"):
    DET = XLMRDetector().load(MODEL_DIR); TRAIN_LOG = json.load(open(f"{WORK}/k06_xlmr_log.json"))
    print("detector loaded from", MODEL_DIR)
else:
    t0 = time.time()
    DET = XLMRDetector().fit(train.text.tolist(), train.a.to_numpy())
    DET.save(MODEL_DIR)
    TRAIN_LOG = {"steps": DET.log, "seconds": round(time.time() - t0), "training_documents": len(train)}
    json.dump(TRAIN_LOG, open(f"{WORK}/k06_xlmr_log.json", "w"))
pc = DET.predict(check.text.tolist())
ENGLISH_CHECK = {"n": len(check), "accuracy": float(((pc > 0.5) == check.a.eq(1).to_numpy()).mean())}
print("English check:", ENGLISH_CHECK)
assert ENGLISH_CHECK["accuracy"] > 0.8, "the detector did not train: send this output back"

In [ ]:
VERSIONS = package_versions("d2vi", "transformers", "torch", "scikit-learn")

def build_multilingual(name, docs):
    scores = score_multilingual(docs, DET, SENT, DET.tok, f"{WORK}/k06_{name}_scores_cache.csv")
    table = docs.merge(scores, on="doc_id", validate="one_to_one")
    feats, info = finalize_multilingual(table)
    feats.to_parquet(f"{WORK}/{name}_features.parquet", index=False)
    ops = operating_points(feats)
    summary = {"corpus": name, "documents": len(docs), **info, "detector_training": TRAIN_LOG,
               "english_check": ENGLISH_CHECK, "operating_points": ops.to_dict("records"),
               "input_sha256": HASHES, "versions": VERSIONS,
               "features_sha256": sha(f"{WORK}/{name}_features.parquet")}
    json.dump(summary, open(f"{WORK}/{name}_summary.json", "w"), indent=1, default=float)
    print(ops.round(3).to_string(index=False))
    return feats

for name in ("m4", "hc3zh"):
    if STATUS[name] != "ready":
        print("skipped", name); continue
    if os.path.exists(f"{WORK}/{name}_features.parquet"):
        print("done already:", name); STATUS[name] = "built"; continue
    build_multilingual(name, m4 if name == "m4" else hc3zh)
    STATUS[name] = "built"

In [ ]:
# DAIGT essays (English): same pipeline as RAID and Beemo in K05; Fast-DetectGPT is the long part
DET = SENT = None
torch.cuda.empty_cache()
from d2vi.features.heldout import score_documents, finalize_features
if STATUS["daigt"] == "ready" and not os.path.exists(f"{WORK}/daigt_features.parquet"):
    scores = score_documents(daigt, SUBJ_DIR, f"{WORK}/k06_daigt_fdgpt_cache.csv")
    table = daigt.merge(scores, on="doc_id", validate="one_to_one")
    sizes = table[table.a.eq(1)].groupby("source").size()
    TRAIN_SOURCE = sorted(sizes[sizes.eq(sizes.max())].index)[0]      # largest machine source; ties: first by name
    feats, info = finalize_features(table, "split", table.source.eq(TRAIN_SOURCE), strata=("source",))
    feats.to_parquet(f"{WORK}/daigt_features.parquet", index=False)
    ops = operating_points(feats, by=("source",))
    summary = {"corpus": "daigt", "documents": len(daigt), **info, "tfidf_trained_on": TRAIN_SOURCE,
               "operating_points": ops.to_dict("records"), "input_sha256": HASHES,
               "versions": package_versions("d2vi", "transformers", "torch", "vaderSentiment", "textstat", "nrclex", "scikit-learn"),
               "features_sha256": sha(f"{WORK}/daigt_features.parquet")}
    json.dump(summary, open(f"{WORK}/daigt_summary.json", "w"), indent=1, default=float)
    print(ops.round(3).to_string(index=False))
    STATUS["daigt"] = "built"
elif os.path.exists(f"{WORK}/daigt_features.parquet"):
    STATUS["daigt"] = "built"

In [ ]:
print("RESULT")
for k, v in STATUS.items():
    print(f"  {k:6s} {v}")
print(sorted(f for f in os.listdir(WORK) if f.endswith(("_features.parquet", "_summary.json"))))